# Homogeneous ReLU Mean-Field Flow

This notebook generates `fig:gradflow-mlp-homogeneous-relu`.  In homogeneous two-layer models, neurons are particles in parameter space and useful teacher features define preferred rays.  The figure shows both particle transport in parameter space and the final concentration of neuron directions.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from matplotlib.collections import LineCollection

NAME = "gradflow-mlp-homogeneous-relu"
out = figure_dir(NAME)
rng = np.random.default_rng(31)

The synthetic ODE below is not a training benchmark.  It is a readable support-propagation picture: angles rotate toward teacher rays while radii adapt to teacher strength.

In [2]:
teacher_angles = np.array([-1.18, 0.08, 1.28])
teacher_strength = np.array([1.08, 0.86, 1.00])
n = 42
angles0 = rng.uniform(-1.62, 1.62, n)
r0 = rng.uniform(0.18, 0.66, n)
X0 = np.column_stack([r0*np.cos(angles0), r0*np.sin(angles0)])
steps = 105
dt = 0.052
traj = np.zeros((steps+1, n, 2))
traj[0] = X0
angles = angles0.copy()
r = r0.copy()

def wrap(a):
    return (a + np.pi) % (2*np.pi) - np.pi

for k in range(steps):
    diff = wrap(angles[:, None] - teacher_angles[None, :])
    j = np.argmin(np.abs(diff), axis=1)
    target = teacher_angles[j]
    strength = teacher_strength[j]
    angles += dt * (-1.85 * wrap(angles - target))
    r += dt * (0.92 * strength - 0.38 * r)
    r = np.clip(r, 0.04, 1.38)
    traj[k+1] = np.column_stack([r*np.cos(angles), r*np.sin(angles)])
final_angles = np.arctan2(traj[-1,:,1], traj[-1,:,0])
all_points = np.vstack([traj.reshape(-1, 2), np.column_stack([1.32*np.cos(teacher_angles), 1.32*np.sin(teacher_angles)])])
xlim, ylim = padded_limits(all_points, pad=0.10)

In [3]:
fig, ax = plt.subplots(figsize=(2.38, 2.18))
for th, s in zip(teacher_angles, teacher_strength):
    ray = np.array([[0, 0], [1.30*s*np.cos(th), 1.30*s*np.sin(th)]])
    ax.plot(ray[:,0], ray[:,1], color=BLUE, lw=1.15, alpha=0.72, zorder=1)
    ax.scatter([ray[-1,0]], [ray[-1,1]], s=DIRAC_MARKER_SIZE * 0.64, marker="o", color=BLUE, edgecolor="none", linewidth=0, zorder=3)
for i in range(n):
    pts = traj[:, i]
    segments = np.stack([pts[:-1], pts[1:]], axis=1)
    cols = [(*interp_color(k/(steps-1)), 0.34) for k in range(steps)]
    ax.add_collection(LineCollection(segments, colors=cols, linewidths=0.50, zorder=2))
ax.scatter(traj[0,:,0], traj[0,:,1], s=DIRAC_MARKER_SIZE * 0.43, marker="o", color=RED, edgecolor="none", linewidth=0, zorder=4)
ax.scatter(traj[-1,:,0], traj[-1,:,1], s=DIRAC_MARKER_SIZE * 0.50, marker="o", color=VIOLET, edgecolor="none", linewidth=0, zorder=5)
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_aspect("equal")
remove_axes(ax)
save_pdf(fig, out / "neuron-trajectories.pdf", pad_inches=0.055)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.38, 1.68))
bins = np.linspace(-1.65, 1.65, 23)
ax.hist(angles0, bins=bins, density=True, color=RED, alpha=0.22, edgecolor="none")
ax.hist(final_angles, bins=bins, density=True, color=VIOLET, alpha=0.52, edgecolor="none")
for th in teacher_angles:
    ax.axvline(th, color=BLUE, lw=1.05, alpha=0.75)
ax.set_xlim(-1.65, 1.65)
ax.set_ylim(0, None)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(labelsize=7, pad=1.5)
box_axes(ax)
save_pdf(fig, out / "direction-histogram.pdf", pad_inches=0.045)
plt.close(fig)